# Okike Academy — Studio v5 Auto-Sync Cloud Video Worker (L4 GPU)

Set `SERVER_URL` to your live Vercel URL (or Cloudflare quick tunnel) and click **Run all** (`Runtime -> Change runtime type -> L4 GPU`).

In [ ]:
SERVER_URL = "https://colour-odds-black-glenn.trycloudflare.com"  # Replace with your permanent Vercel URL once deployed!

!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 > /dev/null
!pip -q install kokoro-onnx soundfile openai-whisper diffusers transformers accelerate omegaconf einops opencv-python mediapipe insightface onnxruntime-gpu kornia scenedetect ffmpeg-python decord huggingface_hub DeepCache face_alignment av lpips requests

import os, subprocess, time
from huggingface_hub import hf_hub_download

if not os.path.exists("LatentSync"):
    !git clone -q https://github.com/bytedance/LatentSync.git
os.makedirs("LatentSync/checkpoints/whisper", exist_ok=True)
hf_hub_download("ByteDance/LatentSync-1.5", "latentsync_unet.pt", local_dir="LatentSync/checkpoints")
hf_hub_download("ByteDance/LatentSync-1.5", "whisper/tiny.pt", local_dir="LatentSync/checkpoints")

!wget -q -nc https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
!wget -q -nc https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin

!mkdir -p remotion_studio/src remotion_studio/public
!curl -sSL "{SERVER_URL}/api/worker/remotion/package.json" -o remotion_studio/package.json
!curl -sSL "{SERVER_URL}/api/worker/remotion/index.ts" -o remotion_studio/src/index.ts
!curl -sSL "{SERVER_URL}/api/worker/remotion/Root.tsx" -o remotion_studio/src/Root.tsx
!curl -sSL "{SERVER_URL}/api/worker/remotion/OkikeLessonVideo.tsx" -o remotion_studio/src/OkikeLessonVideo.tsx
!curl -sSL "{SERVER_URL}/api/worker/mascot/host_1.mp4" -o host_1.mp4
!curl -sSL "{SERVER_URL}/api/worker/mascot/host_2.mp4" -o host_2.mp4
!curl -sSL "{SERVER_URL}/api/worker/script-v5" -o render_explainer_v5.py
!cd remotion_studio && npm install --silent

subprocess.run(["pkill", "-f", "render_explainer_v5.py"], check=False)
log = open("v5_worker.log", "w")
p = subprocess.Popen(["python", "-u", "render_explainer_v5.py", "--server-url", SERVER_URL, "--watch"], stdout=log, stderr=subprocess.STDOUT)
time.sleep(4)
print("Okike Studio v5 Worker LIVE! PID:", p.pid, "| Alive:", p.poll() is None)
!tail -n 20 v5_worker.log